# Example 11.3 / 1: Fitting an Exponential Response

Companion notebook for **11.3 Nonlinear Expressions and Functions** in the AMPL lab Documentation, section "Fitting an Exponential Response". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Fitting an Exponential Response

The parameter `target` is dimensionless, and `x` is bounded to the calibrated range, so `exp(x)` stays safe to evaluate. This one-variable example has a zero residual at $x=\log(4)$.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
param target > 0;     # value exp(x) should match
param lower;          # lower bound on x
param upper > lower;  # upper bound on x

var x >= lower, <= upper, := 1;  # bounded, started at 1

minimize SquaredError:  # squared misfit of exp(x)
    (exp(x) - target)^2;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.param["target"] = 4
ampl.param["lower"] = -2
ampl.param["upper"] = 3

In [5]:
# run.py
ampl.solve(solver="ipopt", verbose=False)
print(ampl.solve_result)
print(round(ampl.var["x"].value(), 6))
print(round(ampl.obj["SquaredError"].value(), 10))

solved
1.386294
0.0


### Pure AMPL: data.dat and model.run

Pure AMPL:

In [6]:
%%writefile data.dat
param target := 4;
param lower := -2;
param upper := 3;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver ipopt;
solve > ipopt.log;
display x, SquaredError;

 
Ipopt 3.12.13: Optimal Solution Found

suffix ipopt_zU_out OUT;
suffix ipopt_zL_out OUT;
x = 1.38629
SquaredError = 1.03245e-20

